# Análisis e Inspección del Registro de Turismo (VUT y Hoteles)

Este notebook realiza la inspección inicial y cálculo de métricas clave para la etapa de limpieza de datos de:
1. **Viviendas de Uso Turístico (VUT / HUT)** procedentes del Registro Oficial de la Generalitat de Catalunya.
2. **Hoteles y Apartamentos Turísticos** de la provincia de Barcelona ([hoteles_y_apartaments_turistics_provincia_barcelona.csv](../data/raw/registre_turisme/hoteles_y_apartaments_turistics_provincia_barcelona.csv)).

## 1. Viviendas de Uso Turístico (VUT / HUT)

Analizamos el archivo oficial de la Generalitat `hut_provincia_barcelona.csv` y complementamos la información de plazas con `opendata_bcn_hut_2016-2026Q1.csv`.

In [1]:
import pandas as pd
import numpy as np

# Carga del registro oficial VUT de la Generalitat
path_vut_gen = '../../data/raw/registre_turisme/hut_provincia_barcelona.csv'
df_vut_gen = pd.read_csv(path_vut_gen)

total_vut_gen = len(df_vut_gen)
print(f"--- VUT REGISTRO OFICIAL (GENERALITAT) ---")
print(f"Cantidad total de registros VUT: {total_vut_gen:,}")

--- VUT REGISTRO OFICIAL (GENERALITAT) ---
Cantidad total de registros VUT: 23,975


In [2]:
# Análisis de Empresas y Titulares
# Separar personas jurídicas (empresas con CIF/Razón Social) de personas físicas (datos protegidos por RGPD -> 'No aplica')
df_vut_empresas = df_vut_gen[df_vut_gen['cif'] != 'No aplica']

total_registros_empresas = len(df_vut_empresas)
total_empresas_unicas = df_vut_empresas['cif'].nunique()
total_particulares = total_vut_gen - total_registros_empresas

print(f"Total registros gestionados por empresas jurídicas (con CIF): {total_registros_empresas:,}")
print(f"Total empresas jurídicas únicas: {total_empresas_unicas:,}")
print(f"Total registros de personas físicas / particulares ('No aplica'): {total_particulares:,}")

print("\n--- Top 10 Empresas con más licencias VUT ---")
top_empresas_vut = df_vut_empresas.groupby(['cif', 'ra_social_del_titular']).size().reset_index(name='vut_count')
top_empresas_vut = top_empresas_vut.sort_values(by='vut_count', ascending=False)
display(top_empresas_vut.head(10))

Total registros gestionados por empresas jurídicas (con CIF): 8,726
Total empresas jurídicas únicas: 2,116
Total registros de personas físicas / particulares ('No aplica'): 15,249

--- Top 10 Empresas con más licencias VUT ---


,cif,ra_social_del_titular,vut_count
197,B08109563,INMOBILIARIA GALLARDO SL,153
14,A08139339,"EDIFICACIONES DEL LITORAL, S.A.",86
1882,B67467589,SINGH PROPCO III SL,80
824,B61903712,VENPRE SL,74
968,B62607668,LLOGUERING SL,73
1560,B66060583,"Masamijomi Immobles, S.L",69
1671,B66612482,APARTAMENTOS COSMOPOLITA SL,67
358,B36009611,ROBECO INVEST SL,65
1948,B86413911,"VESTUARI RENTALS, S.L.",60
967,B62601273,GURB-EUROPA SL,57


In [16]:
# Distribución de cuántos VUT tiene cada empresa jurídica
distribucion_vut = top_empresas_vut['vut_count'].value_counts().sort_index()
df_dist_vut = pd.DataFrame({
    'VUTs por empresa': distribucion_vut.index,
    'Cantidad de empresas': distribucion_vut.values
})
print("--- Distribución del número de VUTs por empresa (primeras 10 filas) ---")
display(df_dist_vut.tail(10))

--- Distribución del número de VUTs por empresa (primeras 10 filas) ---


,VUTs por empresa,Cantidad de empresas
46,57,1
47,60,1
48,65,1
49,67,1
50,69,1
51,73,1
52,74,1
53,80,1
54,86,1
55,153,1


In [4]:
# Plazas en VUT
# El registro oficial de la Generalitat no publica el número de plazas para HUT.
# Se consulta OpenData BCN para la ciudad de Barcelona donde sí se desglosa el número de plazas.
path_vut_bcn = '../../data/raw/vut/opendata_bcn_hut_2016-2026Q1.csv'
df_vut_bcn = pd.read_csv(path_vut_bcn)

# Conversión a int explícito de las plazas
total_plazas_bcn = int(df_vut_bcn['NUMERO_PLACES'].dropna().astype(int).sum())
print(f"--- VUT OPENDATA BCN (CIUDAD DE BARCELONA) ---")
print(f"Total registros VUT Barcelona ciudad: {len(df_vut_bcn):,}")
print(f"Cantidad total de plazas VUT (int): {total_plazas_bcn:,}")

--- VUT OPENDATA BCN (CIUDAD DE BARCELONA) ---
Total registros VUT Barcelona ciudad: 10,718
Cantidad total de plazas VUT (int): 61,899


## 2. Hoteles y Apartamentos Turísticos

Analizamos el dataset unificado de establecimientos hoteleros y apartamentos turísticos en la provincia de Barcelona (`hoteles_y_apartaments_turistics_provincia_barcelona.csv`).

In [5]:
# Carga de datos de hoteles y apartamentos turísticos
path_hoteles = '../../data/raw/registre_turisme/hoteles_y_apartaments_turistics_provincia_barcelona.csv'
df_hoteles = pd.read_csv(path_hoteles)

total_hoteles = len(df_hoteles)
print(f"--- HOTELES Y APARTAMENTOS TURÍSTICOS ---")
print(f"Cantidad total de establecimientos: {total_hoteles:,}")

print("\nDesglose por tipo de establecimiento:")
print(df_hoteles['tipus_establiment'].value_counts())

--- HOTELES Y APARTAMENTOS TURÍSTICOS ---
Cantidad total de establecimientos: 1,562

Desglose por tipo de establecimiento:
tipus_establiment
Hotels                   1442
Apartaments Turístics     120
Name: count, dtype: int64


In [17]:
# Análisis de Empresas Hoteleras y Establecimientos que gestionan
df_hoteles_empresas = df_hoteles[df_hoteles['cif'] != 'No aplica']
total_empresas_hoteles = df_hoteles_empresas['cif'].nunique()

print(f"Cantidad total de empresas hoteleras únicas (por CIF): {total_empresas_hoteles:,}")

top_empresas_hoteles = df_hoteles_empresas.groupby(['cif', 'ra_social_del_titular']).size().reset_index(name='hoteles_count')
top_empresas_hoteles = top_empresas_hoteles.sort_values(by='hoteles_count', ascending=False)

print("\n--- Top 10 Cadenas / Empresas Hoteleras por número de establecimientos ---")
display(top_empresas_hoteles.head(20))


Cantidad total de empresas hoteleras únicas (por CIF): 1,041

--- Top 10 Cadenas / Empresas Hoteleras por número de establecimientos ---


,cif,ra_social_del_titular,hoteles_count
41,A08371346,"Essendi Hospitality Spain, S.A.",14
208,B08236127,"Josel, SLU",8
214,B08370827,SUNWAY SL,7
168,A78304516,MELIA HOTELS INTERNACIONAL SA,6
293,B43067172,"SB HOTELS SPAIN\\, S.L.",6
326,B58227877,DUQUES DE BERGARA S.L.U,6
770,B66068362,DDA HOTEL FUND SL,6
339,B58511882,NH HOTELES ESPAÑA S.A. ( Hotel NH BARCELONA C...,5
474,B61914883,"APOLO, S.L.",5
666,B64810286,"KE MAS 2007, S.L.",5


In [7]:
# Distribución de número de hoteles por empresa
distribucion_hoteles = top_empresas_hoteles['hoteles_count'].value_counts().sort_index()
df_dist_hoteles = pd.DataFrame({
    'Establecimientos por empresa': distribucion_hoteles.index,
    'Cantidad de empresas': distribucion_hoteles.values
})
print("--- Distribución de establecimientos por empresa ---")
display(df_dist_hoteles)

--- Distribución de establecimientos por empresa ---


,Establecimientos por empresa,Cantidad de empresas
0,1,911
1,2,97
2,3,20
3,4,12
4,5,3
5,6,4
6,7,1
7,8,1
8,14,1


In [8]:
# Cantidad total de Plazas y Estancias (convertidos explícitamente a int)
total_plazas_hoteles = int(df_hoteles['total_places'].dropna().astype(float).astype(int).sum())
total_estancias_hoteles = int(df_hoteles['total_estances'].dropna().astype(float).astype(int).sum())

print(f"--- CAPACIDAD TOTAL (HOTELES + APARTAMENTOS TURÍSTICOS) ---")
print(f"Cantidad total de plazas (int): {total_plazas_hoteles:,}")
print(f"Cantidad total de estancias / habitaciones (int): {total_estancias_hoteles:,}")

--- CAPACIDAD TOTAL (HOTELES + APARTAMENTOS TURÍSTICOS) ---
Cantidad total de plazas (int): 166,932
Cantidad total de estancias / habitaciones (int): 86,643
